# Radiance Field ERP — Full Pipeline Demonstration

**TCC @ UFRGS — ERP 3D Classification**

**3D Gaussian Splats → Multi-channel ERP via volumetric radiance field sampling**

Unlike the point-cloud approach (which treats each Gaussian *center* as a
weighted point in a single shell bin), this notebook evaluates the actual 3DGS
**radiance field** — a continuous volumetric density function defined by the full
Gaussian covariance matrices — at concentric sphere shells around the object
centroid.

All computation uses `src/preprocessing/radiance_field.py` directly so that
notebook and production code stay in sync.

## Pipeline

```
PLY file
  -> load_gaussian_ply()           (src.preprocessing.ply_loader)
  -> compute_centroid()            (src.preprocessing.radiance_field)
  -> build_ray_directions()        (src.preprocessing.radiance_field)
  -> precompute_gaussian_params()  (src.preprocessing.radiance_field)
  -> compute_shell_radii()         (src.preprocessing.radiance_field, EgoNeRF exp. spacing)
  -> compute_radiance_field_erp()  (src.preprocessing.radiance_field)
  -> (N_shells, H, W) float32 ERP
```

Or use the convenience wrapper: `gaussian_ply_to_erp(ply_path)` → `(8, 256, 512)`.

In [ ]:
import sys
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.ndimage import gaussian_filter
from pathlib import Path
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# ── project root on sys.path ─────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── import from src ──────────────────────────────────────────────────────────
from src.preprocessing.ply_loader import load_gaussian_ply
from src.preprocessing.radiance_field import (
    compute_centroid,
    build_ray_directions,
    compute_shell_radii,
    precompute_gaussian_params,
    compute_radiance_field_erp,
    gaussian_ply_to_erp,
)
from src.preprocessing.dataset import MODELNET10_CATEGORIES

DATA_ROOT    = PROJECT_ROOT / 'gs_data' / 'modelsplat' / 'modelsplat_ply'
H_RF, W_RF   = 128, 256   # radiance field ERP resolution (reduced for speed)
H_GAL, W_GAL = 64, 128    # gallery resolution (faster)
N_SHELLS     = 8
CUTOFF_SIGMA = 3.0         # Gaussian truncation at 3 sigma (CLAUDE.md default)
BATCH_SIZE   = 128         # Gaussians per batch in einsum

MN10_CATEGORIES = list(MODELNET10_CATEGORIES)
np.random.seed(42)
print(f"Radiance Field ERP — H={H_RF}, W={W_RF}, N_SHELLS={N_SHELLS}, cutoff={CUTOFF_SIGMA}sigma")

In [ ]:
# ── Helper: find sample PLY path ──────────────────────────────────────────────

def find_sample(category: str, split: str = 'train', index: int = 0):
    """Return the PLY path for a given category, split, and index."""
    split_dir = DATA_ROOT / category / split
    if not split_dir.exists():
        return None
    objects = sorted(split_dir.iterdir())
    if index >= len(objects):
        return None
    ply = objects[index] / 'point_cloud.ply'
    return ply if ply.exists() else None


# ── Helper: point-cloud ERP (for comparison) ──────────────────────────────────

def point_cloud_erp(
    gs: dict,
    centroid: np.ndarray,
    n_shells: int,
    H: int,
    W: int,
) -> tuple:
    """Naive point-cloud ERP: assign each Gaussian center to one shell.

    Used for comparison only.  Production pipeline uses compute_radiance_field_erp().
    """
    xyz     = gs['xyz']
    opacity = gs['opacity']
    rgb_gs  = gs['rgb']

    rel = xyz - centroid[None, :]
    r_dist = np.linalg.norm(rel, axis=1)

    safe_r = np.where(r_dist < 1e-9, 1e-9, r_dist)
    theta  = np.arctan2(rel[:, 1], rel[:, 0])
    phi    = np.arcsin(np.clip(rel[:, 2] / safe_r, -1.0, 1.0))
    u = ((theta + np.pi) / (2.0 * np.pi) * W).astype(int) % W
    v = np.clip(((np.pi / 2.0 - phi) / np.pi * H).astype(int), 0, H - 1)

    # Linear shell assignment
    r_5, r_95 = np.percentile(r_dist, [5.0, 95.0])
    shell_edges = np.linspace(r_5, r_95, n_shells + 1)
    shell_radii = 0.5 * (shell_edges[:-1] + shell_edges[1:])
    shell_idx   = np.searchsorted(shell_edges[1:-1], r_dist)
    shell_idx   = np.clip(shell_idx, 0, n_shells - 1)

    density_erp = np.zeros((n_shells, H, W), dtype=np.float64)
    for s in range(n_shells):
        mask = shell_idx == s
        if mask.any():
            np.add.at(density_erp[s], (v[mask], u[mask]), opacity[mask])

    color_accum  = np.zeros((H, W, 3), dtype=np.float64)
    weight_accum = np.zeros((H, W),    dtype=np.float64)
    for c in range(3):
        np.add.at(color_accum[:, :, c], (v, u), opacity * rgb_gs[:, c])
    np.add.at(weight_accum, (v, u), opacity)

    safe_w    = np.where(weight_accum > 0, weight_accum, 1.0)
    color_erp = np.clip(color_accum / safe_w[:, :, None], 0.0, 1.0).astype(np.float32)

    return density_erp.astype(np.float32), color_erp, shell_radii


print("Helper functions defined.")

## 1. Load Sample & Precompute

In [ ]:
# Load the 'table' sample and precompute radiance field parameters

sample_ply = find_sample('table', split='train', index=0)
if sample_ply is None:
    candidates = list((DATA_ROOT / 'table' / 'train').rglob('point_cloud.ply'))
    sample_ply = candidates[0] if candidates else None

if sample_ply is None:
    raise FileNotFoundError(f"No table PLY found under {DATA_ROOT}")

print(f"Loading: {sample_ply}")
gs = load_gaussian_ply(sample_ply)

centroid = compute_centroid(gs['xyz'], gs['opacity'])
gs_precomp = precompute_gaussian_params(gs, centroid)

r_dist = gs_precomp['r_dist']
r_5, r_50, r_95 = np.percentile(r_dist, [5, 50, 95])
# EgoNeRF exponential shell radii (CLAUDE.md / radiance_field.py)
shell_radii_rf = compute_shell_radii(r_dist, n_shells=N_SHELLS)

# Estimate relevant Gaussians per shell (for cutoff_sigma=3.0)
relevant_counts = []
for r_s in shell_radii_rf:
    n_rel = np.sum(np.abs(r_dist - r_s) < CUTOFF_SIGMA * gs_precomp['max_scale'])
    relevant_counts.append(n_rel)

print()
print(f"{'Property':<35} {'Value'}")
print("-" * 55)
print(f"{'N Gaussians':<35} {gs['n_gaussians']:,}")
print(f"{'Centroid':<35} [{centroid[0]:.4f}, {centroid[1]:.4f}, {centroid[2]:.4f}]")
print(f"{'r_dist 5th percentile':<35} {r_5:.5f}")
print(f"{'r_dist 50th percentile (median)':<35} {r_50:.5f}")
print(f"{'r_dist 95th percentile':<35} {r_95:.5f}")
print()
print(f"Shell radii (N={N_SHELLS}, EgoNeRF exponential spacing):")
for s_i, (r_s, n_rel) in enumerate(zip(shell_radii_rf, relevant_counts)):
    print(f"  Shell {s_i+1}: r={r_s:.5f}  |  estimated relevant Gaussians: {n_rel:,}")

In [ ]:
# ── Figure: Gaussian cloud + XZ cross-section with shell circles ──────────────

N_SCATTER = min(5000, gs['n_gaussians'])
idx_sub   = np.random.choice(gs['n_gaussians'], N_SCATTER, replace=False)
xyz_sub   = gs['xyz'][idx_sub]
rgb_sub   = gs['rgb'][idx_sub]
opa_sub   = gs['opacity'][idx_sub]
pt_size   = (opa_sub / (opa_sub.max() + 1e-8) * 20).clip(1, 20)

fig = plt.figure(figsize=(16, 6))

ax3d = fig.add_subplot(1, 2, 1, projection='3d')
ax3d.scatter(
    xyz_sub[:, 0], xyz_sub[:, 1], xyz_sub[:, 2],
    c=rgb_sub, s=pt_size, alpha=0.6, linewidths=0,
)
ax3d.scatter([centroid[0]], [centroid[1]], [centroid[2]],
             c='red', s=200, marker='*', zorder=10, label='Centroid')
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.set_title(f'3D Gaussian Cloud — table ({N_SCATTER:,} subsampled)')
ax3d.view_init(elev=20, azim=45)
ax3d.legend(loc='upper right', fontsize=9)

thickness = 0.08 * (gs['xyz'].max() - gs['xyz'].min())
xz_mask   = np.abs(gs['xyz'][:, 1] - centroid[1]) < thickness
xyz_xz    = gs['xyz'][xz_mask]
rgb_xz    = gs['rgb'][xz_mask]
opa_xz    = gs['opacity'][xz_mask]

ax2d = fig.add_subplot(1, 2, 2)
if len(xyz_xz) > 0:
    pt_xz = (opa_xz / (opa_xz.max() + 1e-8) * 15).clip(1, 15)
    ax2d.scatter(xyz_xz[:, 0], xyz_xz[:, 2], c=rgb_xz, s=pt_xz, alpha=0.7, linewidths=0)

angle_arr = np.linspace(0, 2 * np.pi, 300)
for r_s in shell_radii_rf:
    ax2d.plot(
        centroid[0] + r_s * np.cos(angle_arr),
        centroid[2] + r_s * np.sin(angle_arr),
        'r--', linewidth=1.0, alpha=0.7,
    )
ax2d.plot([], [], 'r--', linewidth=1.0, label=f'Shell radii ({N_SHELLS} shells, exp)')
ax2d.scatter([centroid[0]], [centroid[2]], c='red', s=150, marker='*', zorder=10, label='Centroid')
ax2d.set_xlabel('X'); ax2d.set_ylabel('Z')
ax2d.set_title(f'XZ Cross-section with Shell Boundaries\n(|Y - Y_c| < {thickness:.3f})')
ax2d.set_aspect('equal')
ax2d.legend(loc='upper right', fontsize=9)

fig.suptitle(
    f'Gaussian Cloud & EgoNeRF Shell Structure — table (cutoff={CUTOFF_SIGMA}sigma)',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 2. Radiance Field ERP — table

**Note:** This cell calls `compute_radiance_field_erp()` at reduced resolution
(H=128, W=256) for speed.  The training pipeline uses H=256, W=512 by default.

The full-resolution convenience wrapper is:
```python
erp = gaussian_ply_to_erp(ply_path, n_shells=8, H=256, W=512)
```

In [ ]:
# Compute radiance field ERP for the table sample

t0 = time.time()
ray_dirs = build_ray_directions(H_RF, W_RF)

density_rf = compute_radiance_field_erp(
    gs_precomp=gs_precomp,
    centroid=centroid,
    ray_dirs=ray_dirs,
    shell_radii=shell_radii_rf,
    H=H_RF,
    W=W_RF,
    cutoff_sigma=CUTOFF_SIGMA,
    batch_size=BATCH_SIZE,
)

elapsed = time.time() - t0
print(f"Computed in {elapsed:.1f}s")
print(f"density_rf shape  : {density_rf.shape}")
print(f"density_rf min/max: {density_rf.min():.4f} / {density_rf.max():.4f}")

In [ ]:
# ── Figure: Per-shell density maps (2x4 grid) ─────────────────────────────────

n_cols = 4
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 8))
axes = axes.flatten()

for s in range(N_SHELLS):
    ax = axes[s]
    d  = gaussian_filter(density_rf[s], sigma=1)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    r_s = shell_radii_rf[s]
    ax.set_title(f'Shell {s+1} (r={r_s:.3f})', fontsize=10)
    ax.set_xlabel('Azimuth')
    ax.set_ylabel('Elevation')

fig.suptitle(
    'Radiance Field ERP Density Maps — table\n'
    '(each pixel = sum_i opacity_i * exp(-mahal2_i / 2), EgoNeRF exp. shells)',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 3. Convenience Wrapper: `gaussian_ply_to_erp()`

This is the function called by `GaussianERPDataset` during training.
It runs the full pipeline: PLY load → centroid → ray dirs → shell radii → ERP.

In [ ]:
# Demonstrate the full-pipeline convenience wrapper at reduced resolution

t_wrap = time.time()
erp_full = gaussian_ply_to_erp(
    ply_path=sample_ply,
    n_shells=N_SHELLS,
    H=H_RF,
    W=W_RF,
    cutoff_sigma=CUTOFF_SIGMA,
    batch_size=BATCH_SIZE,
)
print(f"gaussian_ply_to_erp() output shape : {erp_full.shape}")
print(f"dtype                              : {erp_full.dtype}")
print(f"value range                        : [{erp_full.min():.4f}, {erp_full.max():.4f}]")
print(f"Elapsed                            : {time.time()-t_wrap:.1f}s")
# Should match density_rf from the step-by-step computation above
print(f"Max abs diff from step-by-step     : {np.abs(erp_full - density_rf).max():.2e}")

## 4. Comparison: Point Cloud vs Radiance Field

In [ ]:
# Compute point-cloud ERP for the same object and plot side-by-side comparison

print("Computing point-cloud ERP...")
t_pc = time.time()
density_pc, color_pc, shell_radii_pc = point_cloud_erp(
    gs, centroid, N_SHELLS, H_RF, W_RF
)
print(f"  Done in {time.time()-t_pc:.2f}s")
print(f"  density_pc shape: {density_pc.shape}")

COMPARE_SHELLS = [0, N_SHELLS // 2 - 1, N_SHELLS - 1]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
row_labels = ['Point Cloud', 'Radiance Field', 'Point Cloud vs RF']

for col_i, s_i in enumerate(COMPARE_SHELLS):
    # Row 0: Point Cloud density
    ax = axes[0, col_i]
    d  = gaussian_filter(density_pc[s_i], sigma=1)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    ax.set_title(f'PC Shell {s_i+1} (r={shell_radii_pc[s_i]:.3f})', fontsize=10)
    ax.set_xlabel('Azimuth'); ax.set_ylabel('Elevation')

    # Row 1: Radiance Field density
    ax = axes[1, col_i]
    d  = gaussian_filter(density_rf[s_i], sigma=1)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    ax.set_title(f'RF Shell {s_i+1} (r={shell_radii_rf[s_i]:.3f})', fontsize=10)
    ax.set_xlabel('Azimuth'); ax.set_ylabel('Elevation')

    # Row 2: Signed difference (normalized)
    ax = axes[2, col_i]
    d_rf = gaussian_filter(density_rf[s_i].astype(np.float64), sigma=1)
    d_pc = gaussian_filter(density_pc[s_i].astype(np.float64), sigma=1)
    rf_norm = d_rf / (d_rf.max() + 1e-8)
    pc_norm = d_pc / (d_pc.max() + 1e-8)
    diff = rf_norm - pc_norm
    vmax = np.abs(diff).max() + 1e-8
    im = ax.imshow(diff, cmap='RdBu_r', aspect='auto', interpolation='bilinear',
                   vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
    ax.set_title(f'Shell {s_i+1} diff (RF - PC, norm.)', fontsize=10)
    ax.set_xlabel('Azimuth'); ax.set_ylabel('Elevation')

# Row labels
for row_i, label in enumerate(row_labels):
    axes[row_i, 0].set_ylabel(f'{label}\nElevation', fontsize=10)

fig.suptitle(
    'Point Cloud vs Radiance Field ERP — table',
    fontsize=14, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 5. Multi-Category Gallery

In [ ]:
# Gallery: compute radiance field ERP for all 10 ModelNet10 categories
# Uses gallery resolution (H_GAL x W_GAL) for speed

gallery_rf_dens  = {}  # category -> density_erp (N_SHELLS, H_GAL, W_GAL)
ray_dirs_gal = build_ray_directions(H_GAL, W_GAL)

print(f"Gallery resolution: H={H_GAL}, W={W_GAL}")
print()

for cat in MN10_CATEGORIES:
    try:
        ply_path = find_sample(cat, split='train', index=0)
        if ply_path is None:
            candidates = list((DATA_ROOT / cat / 'train').rglob('point_cloud.ply'))
            ply_path = candidates[0] if candidates else None
        if ply_path is None:
            print(f"  {cat:<15} NOT FOUND")
            continue

        g_cat    = load_gaussian_ply(ply_path)
        cent_cat = compute_centroid(g_cat['xyz'], g_cat['opacity'])
        gp_cat   = precompute_gaussian_params(g_cat, cent_cat)
        sr_cat   = compute_shell_radii(gp_cat['r_dist'], n_shells=N_SHELLS)

        t_cat = time.time()
        d_cat = compute_radiance_field_erp(
            gs_precomp=gp_cat,
            centroid=cent_cat,
            ray_dirs=ray_dirs_gal,
            shell_radii=sr_cat,
            H=H_GAL,
            W=W_GAL,
            cutoff_sigma=CUTOFF_SIGMA,
            batch_size=BATCH_SIZE,
        )

        gallery_rf_dens[cat] = d_cat
        elapsed_cat = time.time() - t_cat
        print(f"  {cat:<15} done  ({elapsed_cat:.1f}s, {g_cat['n_gaussians']:,} Gaussians)")

    except Exception as exc:
        print(f"  {cat:<15} ERROR: {exc}")

print(f"\nGallery complete: {len(gallery_rf_dens)}/{len(MN10_CATEGORIES)} categories loaded.")

In [ ]:
# ── Figure: 2x5 middle-shell density gallery ──────────────────────────────────

loaded_cats = [c for c in MN10_CATEGORIES if c in gallery_rf_dens]
n_cats = len(loaded_cats)
mid_shell = N_SHELLS // 2

n_cols_gal = 5
n_rows_gal = (n_cats + n_cols_gal - 1) // n_cols_gal

fig, axes = plt.subplots(n_rows_gal, n_cols_gal, figsize=(20, 4 * n_rows_gal))
axes = axes.flatten()

for i, cat in enumerate(loaded_cats):
    ax = axes[i]
    d  = gaussian_filter(gallery_rf_dens[cat][mid_shell].astype(np.float64), sigma=1)
    im = ax.imshow(d, cmap='hot', aspect='auto', interpolation='bilinear')
    ax.set_title(f'{cat}\n(shell {mid_shell+1}/{N_SHELLS})', fontsize=10)
    ax.axis('off')

for i in range(n_cats, len(axes)):
    axes[i].set_visible(False)

fig.suptitle(
    f'Radiance Field Density ERP Gallery — ModelNet10 (middle shell {mid_shell+1} of {N_SHELLS})',
    fontsize=16, fontweight='bold',
)
plt.tight_layout()
plt.show()

## Summary

### What is the Radiance Field ERP?

This notebook implements and demonstrates a **volumetric** multi-channel ERP
representation derived from 3D Gaussian Splats using the functions in
`src/preprocessing/radiance_field.py`.  Unlike the point-cloud approach
(used in `modelsplat_visualization.ipynb`), the radiance field approach
evaluates the actual **continuous density field** at every sample point on
every shell surface using each Gaussian's full covariance.

### Key Mathematical Formula

For a sample point `p` on shell `s` at radius `r_s`:

    p = centroid + r_s * ray_direction(u, v)
    density[s, u, v] = sum_i [ opacity_i * exp(-0.5 * ||Rt_scaled_i @ (p - mu_i)||^2) ]

where `Rt_scaled_i = R_i^T / scale_i` (CLAUDE.md implementation note).

### Shell Spacing

EgoNeRF exponential spacing (Choi et al., CVPR 2023, §3.2):

    r_s = r_near * (r_far / r_near)^(s / (N-1)),  s = 0..N-1

where `r_near` = 5th percentile and `r_far` = 95th percentile of Gaussian
centre distances from the object centroid.

### Production Usage

```python
from src.preprocessing.radiance_field import gaussian_ply_to_erp
from src.preprocessing.dataset import GaussianERPDataset, MODELNET10_CATEGORIES

# Single object
erp = gaussian_ply_to_erp(ply_path, n_shells=8, H=256, W=512)
# -> (8, 256, 512) float32

# Full dataset (with caching)
dataset = GaussianERPDataset(
    data_root='gs_data/modelsplat/modelsplat_ply',
    categories=MODELNET10_CATEGORIES,
    split='train',
    n_shells=8,
    cache_dir='data/processed/modelnet10/radiance_field/',
)
```